In [0]:
from pyspark.sql.functions import *

In [0]:
bronze_payment=spark.read.format("delta").load("s3://retail-lakehouse-ashu/bronze/payments/")

In [0]:
bronze_payment.printSchema()

In [0]:
bronze_payment.groupby(col("payment_id")).count().filter(col("count")>1).display()

In [0]:
bronze_payment.groupBy("payment_id").agg(
    countDistinct("order_id").alias("order_count")
).filter(
    col("order_count") > 1
).show()

In [0]:
bronze_payment.groupBy("payment_id").agg(
    countDistinct("customer_id").alias("customer_count")
).filter(
    col("customer_count") > 1
).show()

In [0]:
fact_payment=bronze_payment.select(col("payment_id"),col("order_id"),col("customer_id"),col("payment_method"),col("amount"),col("payment_timestamp").cast("timestamp"),col("status"),col("refund.eligible").alias("refund_eligible"),col("refund.refund_status").alias("refund_status"),col("ingestion_timestamp"),col("source_system"),col("batch_id"),col("source_file"))

In [0]:
fact_payment.printSchema()

In [0]:
fact_payment.display()

In [0]:
input_payments=bronze_payment.count()
output_payments=fact_payment.count()
print("Input payments: ",input_payments)
print("Output payments: ",output_payments)

In [0]:
fact_payment.write.mode("overwrite").format("delta").save("/mnt/retail-org/fact_payment")
spark.sql("DROP TABLE IF EXISTS fact_payment")
spark.sql("CREATE TABLE fact_payment USING DELTA LOCATION '/mnt/retail-org/fact_payment'")
spark.sql("OPTIMIZE fact_payment ZORDER BY (payment_id)")
spark.sql